# RAG With Llama 2, Ollama, and Langchain

In this tutorial, we will learn how to implement a retrieval-augmented generation (RAG) application using the Llama 2. We’ll learn why Llama 2 is great for RAG, how to download and access Llama 3.1 locally using Ollama, and how to connect to it using Langchain to build the overall RAG application. We will also learn about the different use cases and real-world applications of Llama 2.

Llama 3.1 is a good choice for RAG, a technique that combines retrieval systems with the text-generating abilities of language models to ensure more accurate and relevant outputs. In RAG, a retrieval system first looks through large datasets to find the most relevant information, which the language model then uses to generate the final response. This is particularly useful for tasks like answering questions, building chatbots, and handling information-heavy tasks, where traditional language models might give outdated or irrelevant answers. With its ability to handle up to 128K tokens and support for multiple languages, Llama 3.1 enhances the quality and reliability of AI-generated content in RAG systems.

## Setup 

To set up a RAG application with Llama 3.1, several steps are required. These include downloading the Llama 3.1 model to your local machine, setting up the environment, loading the necessary libraries, and creating a retrieval mechanism. Finally, we’ll combine this with a language model to build a complete application.

1. Download and install Ollama for your operating system: https://ollama.com/download
2. `pip` install the Python library to generate vector embeddings from the model  with `pip install ollama`.

In [ ]:
!pip install langchain langchain_community langchain-openai scikit-learn langchain-ollama

  Using cached ollama-0.4.8-py3-none-any.whl (13 kB)
You should consider upgrading via the '/home/cdchushig/repos_frescos/easy-local-rag/venv310/bin/python -m pip install --upgrade pip' command.
     |████████████████████████████████| 437 kB 153 kB/s eta 0:00:01
     |████████████████████████████████| 43 kB 116 kB/s eta 0:00:01
     |████████████████████████████████| 223 kB 67 kB/s eta 0:00:01
  Using cached grpcio-1.71.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (5.9 MB)
     |████████████████████████████████| 2.5 MB 23 kB/s eta 0:00:016
  Using cached protobuf-5.29.4-cp38-abi3-manylinux2014_x86_64.whl (319 kB)
     |████████████████████████████████| 4.5 MB 140 kB/s eta 0:00:01
You should consider upgrading via the '/home/cdchushig/repos_frescos/easy-local-rag/venv310/bin/python -m pip install --upgrade pip' command.


In [ ]:
!pip install sentence-transformers

You should consider upgrading via the '/home/cdchushig/repos_frescos/easy-local-rag/venv310/bin/python -m pip install --upgrade pip' command.


## Load and prepare documents

The first step in creating your RAG system is to load the documents we want to use as our knowledge base. In this example, we will use web pages as our source. WebBaseLoader is used to fetch content from each URL provided. The resulting nested lists of documents are then combined into a single, flat list called docs_list, giving us a list of documents.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from PyPDF2 import PdfReader
from io import BytesIO
from urllib.parse import urlparse
import requests

urls = [
    "https://raw.githubusercontent.com/RickParra96/my-local-rag/93ae0e5eb07d6ea3c6fb6410090b9dc2ef6c9291/Docs/AccionDePersonal.pdf",
    "https://raw.githubusercontent.com/RickParra96/my-local-rag/93ae0e5eb07d6ea3c6fb6410090b9dc2ef6c9291/Docs/DiarioDeGerente.pdf",
    "https://raw.githubusercontent.com/RickParra96/my-local-rag/93ae0e5eb07d6ea3c6fb6410090b9dc2ef6c9291/Docs/EmailPersonal.pdf",
    "https://raw.githubusercontent.com/RickParra96/my-local-rag/93ae0e5eb07d6ea3c6fb6410090b9dc2ef6c9291/Docs/PlanMejoras.pdf",
    "https://raw.githubusercontent.com/RickParra96/my-local-rag/93ae0e5eb07d6ea3c6fb6410090b9dc2ef6c9291/Docs/ReporteIncidentes.pdf"
]

def load_pdf_from_url(url: str) -> list[Document]:
    response = requests.get(url)
    response.raise_for_status()
    pdf_reader = PdfReader(BytesIO(response.content))
    documents = []
    for page_number, page in enumerate(pdf_reader.pages, start=1):
        text = page.extract_text()
        if not text:
            continue
        documents.append(
            Document(
                page_content=text,
                metadata={"source": url, "page": page_number}
            )
        )
    return documents

def load_url(url: str) -> list[Document]:
    path = urlparse(url).path.lower()
    if path.endswith('.pdf'):
        return load_pdf_from_url(url)
    return WebBaseLoader(url).load()

# Load documents from the URLs (PDFs or text)
docs_list = []
for url in urls:
    docs_list.extend(load_url(url))


## Split documents into chunks

To make the retrieval process more efficient, we divide the documents into smaller chunks using the RecursiveCharacterTextSplitter. This helps the system handle and search the text more effectively. We can set up the text splitter by specifying the chunk size and overlap. For example, in the code below, we are setting up a text splitter with a chunk size of 180 characters, 40 characters of overlap y un orden de separadores jer?rquico.

In [ ]:
# Initialize a text splitter with hierarchical separators and overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True,
)
# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

## Create a vector store

Next, we need to convert the text chunks into embeddings, which are then stored in a vector store, allowing for quick and efficient retrieval based on similarity. To do this, we use HuggingFaceEmbeddings to generate embeddings for each text chunk, which are then stored in an SKLearnVectorStore. The vector store is set up to return the top 4 most relevant documents for any given query by configuring it with as_retriever(k=4).

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

In [ ]:
from langchain_community.vectorstores import SKLearnVectorStore

# Create embeddings for documents and store them in a vector store
vectorstore = SKLearnVectorStore.from_documents(
    documents=doc_splits,
    embedding=embedding_model
)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 30, "lambda_mult": 0.3},
)

## Set up the LLM and prompt template

In this step, we will set up the LLM and create a prompt template to generate responses based on the retrieved documents.

First, we need to define a prompt template that instructs the LLM on how to format its answers. This template tells the model to use the provided documents to answer questions concisely, using a maximum of three sentences. If the model cannot find an answer, it should simply state that it doesn’t know.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# Define the prompt template for the LLM
prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks.
    Use the following documents to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences maximum and keep the answer concise:
    Question: {question}
    Documents: {documents}
    Answer:
    """,
    input_variables=["question", "documents"],
)

Next, we are connecting to the Llama 2 or Llama 3.1 model using ChatOllama from Langchain, which we have configured with a temperature setting of 0 for consistent responses.

In [ ]:
# Initialize the LLM with Llama 3.1 model
llm = ChatOllama(
    model="deepseek-r1:1.5b",#model="qwen3:0.6b",
    temperature=0,
)

Finally, we create a chain that combines the prompt template with the LLM and uses StrOutputParser to ensure the output is a clean, simple string suitable for display.

In [ ]:
# Create a chain combining the prompt template and LLM
rag_chain = prompt | llm | StrOutputParser()

## Integrate the retriever and LLM into a RAG application

In this step, we will combine the retriever and the RAG chain to create a complete RAG application. We will do this by creating a class called RAGApplication that will handle both the retrieval of documents and the generation of answers.

The RAGApplication class has the run method that takes in the user’s question, uses the retriever to find relevant documents, and then extracts the text from those documents. It then passes the question and the document text to the RAG chain to generate a concise answer.

In [ ]:
# Define the RAG application class
class RAGApplication:
    def __init__(self, retriever, rag_chain):
        self.retriever = retriever
        self.rag_chain = rag_chain
    def run(self, question):
        # Retrieve relevant documents
        documents = self.retriever.invoke(question)
        # Extract content from retrieved documents
        doc_texts = "\\n".join([doc.page_content for doc in documents])
        print('Chunks recuperados:')
        for idx, doc in enumerate(documents, 1):
            print(f'Chunk {idx}:\n{doc.page_content}\n' + '-' * 40)
        # Get the answer from the language model
        answer = self.rag_chain.invoke({"question": question, "documents": doc_texts})
        return answer

In [ ]:
# Initialize the RAG application
rag_application = RAGApplication(retriever, rag_chain)
# Example usage
question = "Cómo se llama la empresa y cuál es el perjuicio ocasionado"
answer = rag_application.run(question)
print("Question:", question)
print("Answer:", answer)

Chunks recuperados:
Chunk 1:
Para la próxima, espero que no tenga que escribir en este diario otra reflexión como esta.  
Firma:  
Sebastián Rivera
----------------------------------------
Chunk 2:
considerarán otras medidas más severas, incluidas posibles sanciones adicionales o 
incluso la terminación del contrato laboral.  
 
Conclusión:  
Esta acción de personal busca corregir el comportamiento negligente de Juanito Montero y 
prevenir futuros incidentes similares. El objetivo es garantizar que todos los empleados cumplan 
con los estándares de desempeño de la empresa, especialmente en áreas tan críticas como el 
monitoreo de sistemas que afectan directamente a la disponibilidad de servicios y la operación de 
la empresa.  
 
Firmado:  
Recursos Humanos:  
Carlos Gomez  
carlos.gomez@patitoec.com  
Empleado:  Juanito Montero
----------------------------------------
Chunk 3:
qué pasa". Los recursos y la infraestructura que tenemos son valiosos, y la pérdida de tiempo o 
dinero, por 